# HeartMuLa Studio — free GPU, real web app

This runs the actual HeartMuLa model (the one from the `heartlib` repo) on a free Colab GPU and wraps it in a **Gradio** app — you get a public link you can open in your phone browser, paste lyrics + tags, and get an MP3 back.

**Before you do anything else:** `Runtime` menu (top) → `Change runtime type` → select **T4 GPU** → `Save`. Without this, generation will fail or crawl on CPU.

Then just run the three code cells below, top to bottom (tap the play button on each, wait for it to finish before starting the next).

**Heads up on timing:** the checkpoint download in Step 2 pulls several GB and can take 5–15 minutes depending on Colab's connection that day. Each song generation after that takes a few minutes too — this model generates audio frame-by-frame, it's not instant like Spotify loading. First generation is slowest since it loads the model into GPU memory.

**Also:** free Colab sessions disconnect after a period of inactivity (or after ~12 hours). If that happens your Gradio link dies — just re-run the cells to get a fresh one.

## Step 1 — install heartlib + Gradio

In [ ]:
!git clone https://github.com/HeartMuLa/heartlib.git
%cd heartlib
!pip install -e . -q
!pip install gradio -q
print('Install done.')

## Step 2 — download the model checkpoints (several GB, be patient)

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="HeartMuLa/HeartMuLaGen", local_dir="./ckpt")
snapshot_download(repo_id="HeartMuLa/HeartMuLa-oss-3B-happy-new-year", local_dir="./ckpt/HeartMuLa-oss-3B")
snapshot_download(repo_id="HeartMuLa/HeartCodec-oss-20260123", local_dir="./ckpt/HeartCodec-oss")
print('Checkpoints downloaded.')

## Step 3 — launch the web app

Once this cell is running, scroll down in the output for a line ending in `.gradio.live` — that's your public link. Open it on your phone, same as any website.

In [ ]:
import gradio as gr
import torch
from heartlib import HeartMuLaGenPipeline

pipe = HeartMuLaGenPipeline.from_pretrained(
    "./ckpt",
    device={"mula": torch.device("cuda"), "codec": torch.device("cuda")},
    dtype={"mula": torch.bfloat16, "codec": torch.float32},
    version="3B",
    lazy_load=True,
)

def generate(lyrics, tags, duration_s, topk, temperature, cfg_scale):
    save_path = "/content/output.mp3"
    with torch.no_grad():
        pipe(
            {"lyrics": lyrics, "tags": tags},
            max_audio_length_ms=int(duration_s * 1000),
            save_path=save_path,
            topk=int(topk),
            temperature=float(temperature),
            cfg_scale=float(cfg_scale),
        )
    return save_path

demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Lyrics", lines=16, placeholder="[Verse]\n...\n\n[Chorus]\n..."),
        gr.Textbox(label="Tags (comma-separated, no spaces)", placeholder="piano,happy,pop"),
        gr.Slider(30, 240, value=120, step=10, label="Max duration (seconds)"),
        gr.Slider(1, 100, value=50, step=1, label="Top-k"),
        gr.Slider(0.1, 2.0, value=1.0, step=0.1, label="Temperature"),
        gr.Slider(0.5, 5.0, value=1.5, step=0.1, label="CFG scale"),
    ],
    outputs=gr.Audio(label="Generated track", type="filepath"),
    title="HeartMuLa Studio — LittleRedBigSmile",
    description="Paste lyrics with [Verse]/[Chorus] sections and comma-separated style tags, then hit Submit.",
)

demo.launch(share=True, debug=True)

## Step 4 — save tracks to Drive (do this if you want to clone your own voice onto them)

HeartMuLa's own vocals are a synthesized generic voice — it has no voice cloning built in. To get **your own voice** singing a HeartMuLa track, the real path is: generate here → save to Drive → open the separate Applio notebook linked below → train a model on your own voice once → run this track through Applio's AI cover feature, which separates the vocal, converts it to your trained voice, and remixes it back with the instrumental automatically.

Run this cell after each generation you want to carry over.

In [ ]:
from google.colab import drive
import shutil, os, datetime

drive.mount('/content/drive')
out_dir = '/content/drive/MyDrive/HeartMuLa_outputs'
os.makedirs(out_dir, exist_ok=True)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dest = f'{out_dir}/heartmula_{stamp}.mp3'
shutil.copy('/content/output.mp3', dest)
print(f'Saved to {dest}')

## Notes

- Lyrics and tags can be pasted straight from the **Lyrics & Tags Deck** artifact — copy from there, paste here.
- Casing doesn't matter, the model lowercases everything internally.
- If you hit a CUDA out-of-memory error, lower the duration slider first — long generations use more GPU memory since it's autoregressive.
- If Step 2 fails partway through, just re-run that cell — `snapshot_download` resumes rather than starting over.
- This notebook uses the exact model class (`HeartMuLaGenPipeline`) from the `heartlib` repo you uploaded, just wrapped in a Gradio interface instead of the command-line script.
- For voice cloning, see the companion notebook and guide: **your own voice model is trained separately in Applio**, not in this notebook — that's a different, more mature tool built specifically for voice conversion, and it does the job better than anything hand-rolled here would.